In [7]:
import pandas as pd
import numpy as np 
from pathlib import Path
from os.path import join, getsize
import kaldiio # for use with kaldi type format
import pyarrow as pa
from pyarrow import csv
import json 

In [8]:
path = '/om2/data/public/GigaSpeech/'

split = 'XL'
# split = 'DEV'
audio_path = Path(join(path, 'audio'))
samp_rate = 16000
manifest_path = Path(f"{path}/data/{split}.csv")

In [9]:
manifest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [10]:
manifest = manifest[~manifest.isna().any(axis=1)] # toss nan file here

In [12]:
manifest.isna().any()

wav_filename    False
transcript      False
speaker         False
dtype: bool

In [13]:
segments = pd.read_csv(Path(path,'data', 'segments'), header=None, 
                       names=['wav_filename', 'speaker', 'start', 'end'], sep='\t')

In [14]:
# wavscp_k = kaldiio.load_scp(Path(path,'data','wav.scp').as_posix())
wavscp = pd.read_csv(Path(path,'data','wav.scp'), header=None,
                                 names=['speaker', 'wav_path'], sep='\t',
                                 )

In [15]:
new_man = pd.merge(manifest,segments, on=['wav_filename', 'speaker'])

In [16]:
new_man = pd.merge(new_man, wavscp, on='speaker')

In [17]:
new_man.isna().any()

wav_filename    False
transcript      False
speaker         False
start           False
end             False
wav_path        False
dtype: bool

In [11]:
# parse_opts = csv.ParseOptions(delimiter='\t')
# table_manifest = csv.read_csv(manifest_path, parse_options=parse_opts)

In [12]:
out_path = Path(path,'data','XL_munged.csv')

new_man.to_csv(out_path, index=False)

In [11]:
train_dict = new_man.to_dict(orient='records')

In [16]:
# out_path = Path(path,'data','XL_munged.json')

# with open(out_path, 'w') as fp:
#     json.dump(train_dict, fp)

In [17]:
split = 'DEV'
manifest_path = Path(f"{path}/data/{split}.csv")
manifest = pd.read_csv(manifest_path, sep='\t',
                               usecols=['wav_filename', 'speaker', 'transcript']) 

In [18]:
dev_man = pd.merge(manifest,segments, on=['wav_filename', 'speaker'])

In [19]:
dev_man = pd.merge(dev_man, wavscp, on='speaker')

In [20]:
dev_dict = dev_man.to_dict(orient='records')

In [23]:
dev_man.loc[0,'wav_path']

'/rdma/vast-rdma/om3/data/public/GigaSpeech/audio/podcast/P0000/POD1000000004.wav'

In [24]:
out_path = Path(path,'data','DEV_munged.json')

with open(out_path, 'w') as fp:
    json.dump(dev_dict, fp)

In [28]:
out_path = Path(path,'data','DEV_munged.csv')

dev_man.to_csv(out_path, index=False)